In [13]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]

for t in thresholds:
    predicted = (transcript_df["similarity"] >= t).astype(int)
    accuracy = (predicted == transcript_df["human_label"]).mean()
    precision = precision_score(transcript_df["human_label"], predicted, zero_division=0)
    recall = recall_score(transcript_df["human_label"], predicted, zero_division=0)
    print(f"Threshold {t}: accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

Threshold 0.6: accuracy=0.960  precision=0.977  recall=0.977
Threshold 0.7: accuracy=0.920  precision=0.976  recall=0.930
Threshold 0.75: accuracy=0.900  precision=0.975  recall=0.907
Threshold 0.8: accuracy=0.880  precision=0.974  recall=0.884
Threshold 0.85: accuracy=0.880  precision=1.000  recall=0.860
Threshold 0.9: accuracy=0.840  precision=1.000  recall=0.814


In [15]:
import pandas as pd

transcript_df = pd.read_csv("../data/pipeline_results.csv")
print(transcript_df.shape)
print(transcript_df.columns.tolist())


(50, 9)
['passage', 'question', 'answer', 'answer_word_count', 'audio_path', 'transcript', 'similarity', 'human_label', 'prediction']


In [17]:
from sklearn.metrics import precision_score, recall_score

thresholds = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]

for t in thresholds:
    predicted = (transcript_df["similarity"] >= t).astype(int)
    accuracy = (predicted == transcript_df["human_label"]).mean()
    precision = precision_score(transcript_df["human_label"], predicted, zero_division=0)
    recall = recall_score(transcript_df["human_label"], predicted, zero_division=0)
    print(f"Threshold {t}: accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

Threshold 0.6: accuracy=0.960  precision=0.977  recall=0.977
Threshold 0.7: accuracy=0.920  precision=0.976  recall=0.930
Threshold 0.75: accuracy=0.900  precision=0.975  recall=0.907
Threshold 0.8: accuracy=0.880  precision=0.974  recall=0.884
Threshold 0.85: accuracy=0.880  precision=1.000  recall=0.860
Threshold 0.9: accuracy=0.840  precision=1.000  recall=0.814


In [19]:
def classify(similarity):
    if similarity >= 0.65:
        return "Correct"
    elif similarity <= 0.55:
        return "Incorrect"
    else:
        return "Borderline"

transcript_df["prediction"] = transcript_df["similarity"].apply(classify)
transcript_df.to_csv("../data/pipeline_results.csv", index=False)

print(transcript_df["prediction"].value_counts())

prediction
Correct       41
Incorrect      7
Borderline     2
Name: count, dtype: int64


In [21]:
borderline = transcript_df[transcript_df["prediction"] == "Borderline"]
print(borderline[["answer", "transcript", "similarity", "human_label"]])

               answer          transcript  similarity  human_label
13      Pleurobrachia         Plorobracia    0.639905            1
38  William Stranahan  William Strenohin.    0.647472            1


In [30]:
import pandas as pd

df = pd.read_csv("../data/pipeline_results.csv")
print(df.shape)

errors = df[df["prediction"].isin(["Incorrect", "Borderline"])]
print(errors[["answer", "transcript", "similarity", "prediction"]])

(50, 9)
                     answer                        transcript  similarity  \
2            Lower Lorraine                   Lower the rain.    0.279233   
4                 10,000 m2             10,000 square meters.    0.000000   
10                  Beyoncé                        Be on say.    0.167976   
13            Pleurobrachia                       Plorobracia    0.639905   
27  Mi'kmaq and the Abenaki  Me, Kamek, and I'll be knocking.    0.222933   
32             gas turbines                    Gast her buns.    0.159867   
35                  Thoreau                             Dora.    0.261759   
38        William Stranahan                William Strenohin.    0.647472   
40                 Nintendo                               念獵獵    0.131477   

    prediction  
2    Incorrect  
4    Incorrect  
10   Incorrect  
13  Borderline  
27   Incorrect  
32   Incorrect  
35   Incorrect  
38  Borderline  
40   Incorrect  


In [32]:

errors = df[df["prediction"].isin(["Incorrect", "Borderline"])]
print(errors[["answer", "transcript", "similarity", "prediction"]])

                     answer                        transcript  similarity  \
2            Lower Lorraine                   Lower the rain.    0.279233   
4                 10,000 m2             10,000 square meters.    0.000000   
10                  Beyoncé                        Be on say.    0.167976   
13            Pleurobrachia                       Plorobracia    0.639905   
27  Mi'kmaq and the Abenaki  Me, Kamek, and I'll be knocking.    0.222933   
32             gas turbines                    Gast her buns.    0.159867   
35                  Thoreau                             Dora.    0.261759   
38        William Stranahan                William Strenohin.    0.647472   
40                 Nintendo                               念獵獵    0.131477   

    prediction  
2    Incorrect  
4    Incorrect  
10   Incorrect  
13  Borderline  
27   Incorrect  
32   Incorrect  
35   Incorrect  
38  Borderline  
40   Incorrect  


In [34]:
manifest = pd.read_csv("../data/audio_manifest.csv")
nintendo_row = manifest[manifest["answer"] == "Nintendo"]
print(nintendo_row[["answer", "audio_path"]])

      answer                         audio_path
40  Nintendo  data/audio/pyttsx3/pyttsx3_40.mp3


In [36]:
merged = df.merge(manifest[["answer", "audio_path"]], on="answer", how="left")
merged["tts_engine"] = merged["audio_path"].apply(lambda p: "gtts" if "gtts" in p else "pyttsx3")

print(merged.groupby("tts_engine")["prediction"].value_counts())

KeyError: 'audio_path'

In [38]:
print(merged.columns.tolist())


['passage', 'question', 'answer', 'answer_word_count', 'audio_path_x', 'transcript', 'similarity', 'human_label', 'prediction', 'audio_path_y']


In [40]:
print(manifest.columns.tolist())

['passage', 'question', 'answer', 'answer_word_count', 'audio_path']


In [42]:
merged["tts_engine"] = merged["audio_path_x"].apply(lambda p: "gtts" if "gtts" in p else "pyttsx3")

print(merged.groupby("tts_engine")["prediction"].value_counts())

tts_engine  prediction
gtts        Correct       21
            Incorrect      3
            Borderline     1
pyttsx3     Correct       20
            Incorrect      4
            Borderline     1
Name: count, dtype: int64
